# 02 - Data Preprocessing
Cleans the raw pairs, filters by sentence length, splits into train/val/test, and builds vocabularies.

In [1]:
import sys, pickle
import pandas as pd
from sklearn.model_selection import train_test_split

sys.path.append("../src")
from dataset import Vocabulary

ROLL_NUMBER = 55
SAMPLE_SIZE = 30000
MIN_LEN, MAX_LEN = 2, 15

## Load raw pairs

In [2]:
rows = []
with open("../data/raw/spa.txt", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) >= 2:
            rows.append((parts[0], parts[1]))

df = pd.DataFrame(rows, columns=["english", "spanish"]).drop_duplicates()
print("Raw pairs:", len(df))
df.head()

Raw pairs: 144215


,english,spanish
0,Go.,Ve.
1,Go.,Vete.
2,Go.,Vaya.
3,Go.,Váyase.
4,Go.,Id.


## Clean and filter by length

In [3]:
vocab_helper = Vocabulary()  # just used here for normalize_text()

df["english"] = df["english"].apply(vocab_helper.normalize_text)
df["spanish"] = df["spanish"].apply(vocab_helper.normalize_text)

df["english_length"] = df["english"].apply(lambda s: len(s.split()))
df["spanish_length"] = df["spanish"].apply(lambda s: len(s.split()))

df = df[df["english_length"].between(MIN_LEN, MAX_LEN) & df["spanish_length"].between(MIN_LEN, MAX_LEN)]
print("After cleaning/filtering:", len(df))

After cleaning/filtering: 141451


## Subsample

In [4]:
if len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=ROLL_NUMBER)
df = df.reset_index(drop=True)
print("Final dataset size:", len(df))

Final dataset size: 30000


## Split 80/10/10

In [5]:
train_df, temp_df = train_test_split(df, train_size=0.8, random_state=ROLL_NUMBER)
val_df, test_df = train_test_split(temp_df, train_size=0.5, random_state=ROLL_NUMBER)

print("train:", len(train_df), "val:", len(val_df), "test:", len(test_df))

train: 24000 val: 3000 test: 3000


## Build vocabularies (on train split only)

In [6]:
english_vocab = Vocabulary(min_freq=2)
english_vocab.build_vocabulary(train_df["english"].tolist())

spanish_vocab = Vocabulary(min_freq=2)
spanish_vocab.build_vocabulary(train_df["spanish"].tolist())

print("English vocab size:", len(english_vocab))
print("Spanish vocab size:", len(spanish_vocab))

English vocab size: 4146
Spanish vocab size: 5938


## Save

In [7]:
train_df.to_csv("../data/processed/train.csv", index=False)
val_df.to_csv("../data/processed/validation.csv", index=False)
test_df.to_csv("../data/processed/test.csv", index=False)

pickle.dump(english_vocab, open("../data/processed/english_vocab.pkl", "wb"))
pickle.dump(spanish_vocab, open("../data/processed/spanish_vocab.pkl", "wb"))

print("Saved to data/processed/")

Saved to data/processed/
